# धडा १३ - कॉग्नी नॉलेज ग्राफसह एजंट मेमरी


## सेटअप

हा नोटबुक कसे बांधायचे हे दाखवतो एक बुद्धिमान **कोडिंग सहाय्यक** जो सातत्यपूर्ण स्मृतीसह आहे, [**Cognee**](https://www.cognee.ai/) नॉलेज ग्राफ्ज आणि **Microsoft Agent Framework** (MAF) वापरून.

Cognee असंरचित मजकूराचे रूपांतर संरचित, प्रश्न विचारणारे ज्ञान ग्राफमध्ये करते जो वेक्टर एम्बेडिंग्जने समर्थित आहे — ज्यामुळे तुमच्या एजंटला समृद्ध, नाते-ज्ञानी दीर्घकालीन स्मृती मिळते.

### तुम्हाला काय शिकायला मिळेल
1. **नॉलेज ग्राफ्स तयार करा**: विकसक प्रोफाइल आणि सर्वोत्तम पद्धती संरचित, प्रश्न विचारण्यास योग्य ज्ञानात रूपांतरित करा.
2. **Cognee ला MAF सोबत एकत्रित करा**: `@tool` फंक्शन्स वापरून MAF एजंटला Cognee च्या नॉलेज ग्राफला प्रश्न विचारायला द्या.
3. **सत्र-जागरूक संभाषणे**: एकाच सत्रातील अनेक प्रश्नांमध्ये संदर्भ राखा.
4. **दीर्घकालीन स्मृती**: महत्त्वाचे ज्ञान सत्रांमध्ये सतत ठेवा आणि नवीन संभाषणांमध्ये ते पुनः मिळवा.

### गरजांची यादी
- Python 3.9+
- स्थानिकपणे Redis चालू (`docker run -d -p 6379:6379 redis`) सत्र व्यवस्थापनासाठी
- LLM API की (उदा. OpenAI) — `.env` फाईलमध्ये `LLM_API_KEY` सेट करा
- `.env` फाईलमध्ये `CACHING=true` (Cognee सत्रांसाठी आवश्यक)
- Azure AI Foundry प्रोजेक्ट ज्यामध्ये चॅट मॉडेल डिप्लॉय केलेले आहे
- `.env` मध्ये `AZURE_AI_PROJECT_ENDPOINT` आणि `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI मध्ये प्रमाणीकृत (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

##代理ची आठवण प्रकार

हा नोटबुक मुख्य लेसन 13 नोटबुकमधील तीच तीन आठवण प्रकार तपासतो, परंतु लांब-कालावधीतील आठवण बॅकएंड म्हणून Cognee वापरतो:

| आठवण प्रकार | यंत्रणा | आयुष्यकाल |
|---|---|---|
| **काम करणारी** | `agent.create_session()` (MAF) | एक संवाद |
| **लहान-कालावधी** | Cognee सत्र कॅश (Redis) | एक सत्र |
| **लांब-कालावधी** | Cognee ज्ञान ग्राफ + व्हेक्टर | कायमस्वरूपी |

### Cognee चे आठवण वास्तुकला
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## कॉगनी स्टोरेज तयार करा


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## भाग 1 — ज्ञान भांडार तयार करणे

आम्ही आमच्या कोडिंग सहाय्यकासाठी सर्वसमावेशक ज्ञान भांडार तयार करण्यासाठी तीन प्रकारचे डेटा घेतो:

1. **विकसक प्रोफाइल** — वैयक्तिक कौशल्ये आणि तांत्रिक पार्श्वभूमी  
2. **पायथॉन सर्वोत्तम पद्धती** — वापरात असलेल्या मार्गदर्शकांसह पायथॉनचे झेन  
3. **ऐतिहासिक संवाद** — विकासक आणि एआय सहाय्यकांमधील भूतकाळातील प्रश्नोत्तरे सत्रं


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## नॉलेज ग्राफचे दृश्यांकन करा

Cognee काढलेल्या घटकांच्या आणि नात्यांच्या एका परस्परसंवादी HTML दृश्यांकनाची सुविधा देऊ शकतो.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## मेमरी समृद्ध करा मेमिफायसह

`memify()` ज्ञान ग्राफचे विश्लेषण करते आणि बुद्धिमान नियम तयार करते — नमुने, सर्वोत्तम पद्धती आणि संकल्पनांमधील नातेसंबंध ओळखून.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## भाग 2 — Cognee टूल्ससह MAF एजंट

आता आपण एक MAF एजंट तयार करतो जो `@tool` फंक्शन्सद्वारे Cognee च्या ज्ञान ग्राफवर क्वेरी करू शकतो. हे एजंटला सत्रांद्वारे संभाषणात्मक संदर्भ राखताना ग्राफ-जाणकार सेमांटिक शोधाची संपूर्ण क्षमता वापरण्याची परवानगी देते.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## सत्रांसह कार्यशील स्मृती

`AgentSession` (जो `agent.create_session()` द्वारे तयार केला जातो) सत्रामध्ये कार्यशील स्मृती प्रदान करतो. एजंट आधीच्या संदेशांकडे पुनः पाहू शकतो तसेच Cognee च्या दीर्घकालीन ज्ञान ग्राफला देखील चौकशी करू शकतो.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## नवीन सत्र — दीर्घकालीन स्मृती टिकून राहते

नवीन सत्र सुरू केल्यावर कार्यस्मृती क्लिअर होते, परंतु Cognee ज्ञान ग्राफ अजूनही उपलब्ध आहे. एजंट पूर्ण नवीन संवादातही तीच दीर्घकालीन ज्ञान परत घेऊ शकतो.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

या नोटबुकमध्ये आपण एक कोडिंग सहाय्यक तयार केला जो **MAF च्या कार्यस्मृती** (`agent.create_session()`) ला **Cognee च्या दीर्घकालीन ज्ञान ग्राफ** सोबत एकत्र करतो.

### आपण काय शिकलात
1. **ज्ञान ग्राफ बांधणी**: Cognee अपरिष्कृत मजकूर ग्रहण करून ग्राफ + व्हेक्टर मेमरी तयार करते.
2. **ग्राफ समृद्धीकरण memify सह**: विद्यमान ग्राफवर आधारित तथ्ये आणि अधिक सखोल संबंध तयार करणे.
3. **MAF + Cognee समाकलन**: `@tool` फंक्शन्स MAF एजंटला Cognee च्या ग्राफवर स्वाभाविकपणे विचारण्यास परवानगी देतात.
4. **कार्यस्मृती + दीर्घकालीन स्मृती**: `AgentSession` (द्वारे `agent.create_session()`) सत्र संदर्भ प्रदान करते, तर Cognee स्थायी ज्ञान पुरवते.
5. **NodeSets सह फिल्टर्ड शोध**: ज्ञान ग्राफमधील विशिष्ट उपसमूहांचा लक्ष्य करा (उदा. फक्त तत्त्वे).

### मुख्य मुद्दे
- **Cognee** अपरिष्कृत मजकूराला संरचित, संबंध-सजग स्मृतीमध्ये रूपांतरित करतो — फ्लॅट व्हेक्टर स्टोअरपेक्षा अधिक सामर्थ्यशाली.
- **`@tool` फंक्शन्स** MAF एजंट्स आणि बाह्य ज्ञान प्रणालींमध्ये स्वच्छ सेतू कार्य करतात.
- **`AgentSession`** (द्वारे `agent.create_session()`) प्रत्येक संभाषणाचा संदर्भ दीर्घकालीन ज्ञानापासून वेगळा ठेवतो.
- तोच ज्ञान ग्राफ अनेक सत्रे आणि एजंटसाठी उपलब्ध असतो.

### वास्तविक जगातील उपयोग
- **विकसक सहाय्यक**: कोड पुनरावलोकन, अपघटन विश्लेषण, आर्किटेक्चर सहाय्यक
- **ग्राहक-सामना सहाय्यक**: उत्पादन कागदपत्रे, FAQ, आणि CRM टीपांवर आधारित समर्थन एजंट
- **आतील तज्ञ सहाय्यक**: धोरण, कायदेशीर, किंवा सुरक्षा सहाय्यक मार्गदर्शक तत्त्वांवर आधारित विचार करू शकतात
- **एकत्रित डेटा स्तर**: संरचित आणि अपरिष्कृत डेटा एकत्रित करून प्रश्न विचारण्यायोग्य ग्राफ तयार करा

### पुढील पावले
- Cognee मध्ये कालिक जागरूकता वापरून प्रयोग करा
- विशिष्ट क्षेत्रासाठी OWL ओण्टॉलॉजी परिभाषित करा
- पुनर्प्राप्ती सुधारण्यासाठी वापरकर्त्याचा अभिप्राय संकलन जोडा
- एकाच Cognee स्मृती स्तर सामायिक करणाऱ्या बहु-एजंट प्रणालींमध्ये प्रमाण वाढवा


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सूचना**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून भाषांतरित करण्यात आला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये चुका किंवा अचूकतेतील त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे होणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थनिर्देशांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
